In [ ]:
import os

os.chdir("..")

In [ ]:
from fsdc_calories import DataCal
import polars as pl

dc = DataCal()

In [ ]:
dc.gen_nuti_data_local()

In [ ]:
dc.gen_graphs_plate()

In [ ]:
pl.read_excel("data/schedule_b_reference.xlsx").write_parquet("data/external/schedule_b_reference")
nutri_matrix = pl.read_parquet("src/fsdc_calories/data/nutri_matrix.parquet")
nutri_matrix.columns

In [ ]:
df_import = dc.gen_nuti_data()
df_import = df_import.with_columns(pl.col("fiscal_year").cast(pl.Int64))
df_import

In [ ]:
df_local = pl.read_excel("data/raw/local_data.xlsx")
df_local.write_parquet("src/fsdc_calories/data/local_nutri_data.parquet")

In [ ]:
df_local = pl.read_excel("data/raw/local_data.xlsx")
nutri_matrix = pl.read_parquet("src/fsdc_calories/data/nutri_matrix.parquet")
nutri_matrix = pl.read_parquet("src/fsdc_calories/data/nutri_matrix.parquet")
nutri_matrix = nutri_matrix.unique(subset="schedule_b", keep="first")
df_local = df_local.select(pl.col("fiscal_year", "schedule_b", "cantidad_convertida_mediana"))
df_local = df_local.with_columns(pl.col("schedule_b").cast(pl.String).str.zfill(4))
df_local = df_local.join(nutri_matrix, on="schedule_b", how="inner", validate="m:1")
df_local = df_local.rename({"schedule_b": "hts_code"})


df_local = df_local.with_columns(
    total_calories=pl.col("cantidad_convertida_mediana") * pl.col("calories"),
    total_fats=pl.col("cantidad_convertida_mediana") * pl.col("fats"),
    total_sugars=pl.col("cantidad_convertida_mediana") * pl.col("sugars"),
    total_protein=pl.col("cantidad_convertida_mediana") * pl.col("protein"),
    total_saturated_fat_g=pl.col("cantidad_convertida_mediana") * pl.col("saturated_fat_g"),
    total_cholesterol_mg=pl.col("cantidad_convertida_mediana") * pl.col("cholesterol_mg"),
    total_sodium_mg=pl.col("cantidad_convertida_mediana") * pl.col("sodium_mg"),
    total_carbohydrate_g=pl.col("cantidad_convertida_mediana") * pl.col("carbohydrate_g"),
    total_fiber_g=pl.col("cantidad_convertida_mediana") * pl.col("fiber_g"),
    total_sugar_g=pl.col("cantidad_convertida_mediana") * pl.col("sugar_g"),
    total_vitamin_d_iu=pl.col("cantidad_convertida_mediana") * pl.col("vitamin_d_iu"),
    total_calcium_mg=pl.col("cantidad_convertida_mediana") * pl.col("calcium_mg"),
    total_potassium_mg=pl.col("cantidad_convertida_mediana") * pl.col("potassium_mg"),
    total_iron_mg=pl.col("cantidad_convertida_mediana") * pl.col("iron_mg"),
)
cols = [
    "total_calories",
    "total_fats",
    "total_sugars",
    "total_protein",
    "total_saturated_fat_g",
    "total_cholesterol_mg",
    "total_sodium_mg",
    "total_carbohydrate_g",
    "total_fiber_g",
    "total_sugar_g",
    "total_vitamin_d_iu",
    "total_calcium_mg",
    "total_potassium_mg",
    "total_iron_mg",
]
df_local = df_local.with_columns(
    **{f"{x}_ecdf": pl.col(x).rank("ordinal") / pl.len() for x in cols}
)

df_local = df_local.group_by(["fiscal_year", "hts_code"]).agg(pl.col(cols).sum())

df_local = df_local.filter(pl.col("fiscal_year") > 2009)
df_local 

In [ ]:
import polars as pl
import altair as alt

# 1. Find the HTS codes that exist in BOTH dataframes (Intersection)
shared_hts = set(df_import['hts_code']).intersection(set(df_local['hts_code']))

df_import_filtered = (
    df_import
    .filter(pl.col('hts_code').is_in(shared_hts))
    .with_columns(pl.lit('Import').alias('source_type'))
)

# A robust renaming map that strips out _g, _mg, _iu before adding _kg
rename_mapping = {}
for col in df_local.columns:
    if col.startswith("total_") and col != "total_calories" and not col.endswith("_kg"):
        # Strip old unit suffixes from the base name if they exist
        clean_name = col
        for suffix in ["_g", "_mg", "_iu"]:
            if clean_name.endswith(suffix):
                clean_name = clean_name[:-len(suffix)]
                break
        rename_mapping[col] = f"{clean_name}_kg"

df_local_filtered = (
    df_local
    .filter(pl.col('hts_code').is_in(shared_hts))
    .rename(rename_mapping)  # Now perfectly aligns "total_saturated_fat_g" -> "total_saturated_fat_kg"
    .with_columns(pl.lit('Local').alias('source_type'))
)
# 3. Concatenate the filtered Polars dataframes
source_df = pl.concat([df_import_filtered, df_local_filtered])

# 4. Dynamically find the min and max fiscal years for the slider
min_year = int(source_df['fiscal_year'].min())
max_year = int(source_df['fiscal_year'].max())

# 5. Create the slider and selection point for 'fiscal_year'
slider = alt.binding_range(min=min_year, max=max_year, step=1, name="Select Year: ")
select_year = alt.selection_point(name='fiscal_year', fields=['fiscal_year'],
                                   bind=slider, value=max_year)

# 6. Define a cohesive, professional color palette
color_scale = alt.Scale(
    domain=['Import', 'Local'],
    range=['#1f4e79', '#2d8a81']
)

# 7. Define the base chart configuration with clean typography and % calculations
base = alt.Chart(source_df).add_params(
    select_year
).transform_filter(
    select_year
).transform_calculate(
    # Step A: Isolate local calories per row BEFORE any chart-level filtering happens
    is_local_cal=alt.expr.if_(alt.datum.source_type == 'Local', alt.datum.total_calories, 0)
).transform_joinaggregate(
    # Step B: Sum the isolated local calories globally per HTS code/year block
    global_local_calories='sum(is_local_cal)',
    hts_total='sum(total_calories)',
    groupby=['hts_code', 'fiscal_year']
).transform_joinaggregate(
    source_hts_total='sum(total_calories)',
    groupby=['hts_code', 'source_type', 'fiscal_year']
).transform_calculate(
    # Calculate the percentage contribution per year block
    pct=alt.datum.source_hts_total / alt.datum.hts_total
).properties(
    width=280,
    height=alt.Step(20) 
)

# FIXED SORTING LOGIC: Uses the global aggregated 'global_local_calories' field
# Sorting order 'descending' places the highest local calorie categories at the top.
y_sort_axis = alt.Y('hts_code:O').sort(field='global_local_calories', op='max', order='descending')

# --- CHART COLUMNS ---

# Left Chart: Import data 
left_base = base.transform_filter(alt.datum.source_type == 'Import')

left_bars = left_base.encode(
    y=y_sort_axis.axis(None), 
    x=alt.X('sum(total_calories):Q')
        .title('Total Calories')
        .sort('descending')
        .axis(format='~s', grid=True, labelExpr="replace(datum.label, 'G', 'B')"), 
    color=alt.Color('source_type:N').scale(color_scale).legend(None)
).mark_bar()

left_text = left_base.encode(
    y=y_sort_axis.axis(None),
    x=alt.X('sum(total_calories):Q').sort('descending'),
    text=alt.Text('max(pct):Q', format='.1%')
).mark_text(
    dx=5,
    align='right', 
    baseline='middle',
    font='Arial',
    fontSize=10,
    color='#333333'
)

left = alt.layer(left_bars, left_text).properties(
    title=alt.TitleParams('Imported Calories', anchor='end', font='Arial', fontSize=13, color='#333333')
)


# Middle Chart: Centralized HTS Code Text Labels
middle = base.transform_aggregate(
    global_local_calories='max(global_local_calories)', # Bind sorting metric to middle channel
    groupby=['hts_code']
).encode(
    y=y_sort_axis.axis(None),
    text=alt.Text('hts_code:O'),
).mark_text(font='Arial', fontSize=11, color='#555555').properties(width=60)


# Right Chart: Local data
right_base = base.transform_filter(alt.datum.source_type == 'Local')

right_bars = right_base.encode(
    y=y_sort_axis.axis(None),
    x=alt.X('sum(total_calories):Q')
        .title('Total Calories')
        .axis(format='~s', grid=True, labelExpr="replace(datum.label, 'G', 'B')"),
    color=alt.Color('source_type:N').scale(color_scale).legend(None)
).mark_bar()

right_text = right_base.encode(
    y=y_sort_axis.axis(None),
    x=alt.X('sum(total_calories):Q'),
    text=alt.Text('max(pct):Q', format='.1%')
).mark_text(
    dx=5,
    align='left',
    baseline='middle',
    font='Arial',
    fontSize=10,
    color='#333333'
)

right = alt.layer(right_bars, right_text).properties(
    title=alt.TitleParams('Local Calories', anchor='start', font='Arial', fontSize=13, color='#333333')
)


# 8. Concatenate and apply top-level chart configurationsf
chart = alt.concat(
    left, middle, right, spacing=10
).properties(
    title=alt.TitleParams(
        text='Caloric Distribution Comparison',
        subtitle=['Comparing localized and imported total calories across shared HTS codes.', ' '],
        fontSize=16,
        anchor='middle',  
        font='Arial',
        subtitleColor='#666666'
    )
).configure_view(
    stroke=None 
).configure_axis(
    domain=False,
    ticks=False,
    labelFont='Arial',
    titleFont='Arial',
    titleColor='#444444'
)

chart

In [ ]:
df_local.filter((pl.col("hts_code") == "0707") & (pl.col("fiscal_year") == 2020))

In [ ]:
df_import.filter((pl.col("hts_code") == "0707") & (pl.col("fiscal_year") == 2020))